In [1]:
import os
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
def recognize_captcha_via_prepostseo(driver, wait, image_path):
    #Открываем новую вкладку и переключаемся на неё
    driver.execute_script("window.open('');")
    driver.switch_to.window(driver.window_handles[-1])
    driver.get("https://www.prepostseo.com/ru/image-to-text")
    
    #Ожидаем загрузки поля для отправки файла и передаем путь к картинке
    file_input = wait.until(EC.presence_of_element_located(( By.XPATH, "//input[@type='file' or @id='file-upload']")))
    absolute_path = os.path.abspath(image_path)
    file_input.send_keys(absolute_path)
    
    #Находим кнопку отправки и нажимаем
    submit_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[contains(text(), 'Извлечь текст')] | //button[contains(., 'Извлечь текст')]")))
    submit_btn.click()
    
    #Ожидаем появления ТЕКСТА внутри поля результата
    result_locator = (By.XPATH, "//*[@id='result__text0']")
    
    # Ждем пока элемент просто появится 
    result_textarea = wait.until(EC.presence_of_element_located(result_locator))
    
    #ожидание заполнения поля текстом (максимум 15 секунд)
    recognized_text = ""
    for _ in range(15):
        # Пробуем получить через .text, innerText или value
        text_variant = result_textarea.get_attribute("value") or result_textarea.get_attribute("innerText") or result_textarea.text
        if text_variant and text_variant.strip():
            recognized_text = text_variant.strip()
            break
        time.sleep(1) # Короткая пауза перед повторной проверкой
    
    #Закрываем вкладку сервиса и возвращаемся на исходную страницу
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    
    return recognized_text


In [3]:
# Создаем папку для временного хранения капчи, если её нет
os.makedirs('capcha_temp', exist_ok=True)

options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

try:
    # 1. Переходим на страницу RuCaptcha
    driver.get("https://rucaptcha.com/demo/normal")
    
    # идентификатор первой вкладки
    demo_window = driver.current_window_handle
    
    #Находим и сохраняем капчу
    captcha_element = wait.until(EC.visibility_of_element_located(( By.XPATH, "//img[starts-with(@class, '_captchaImage')]")))
    captcha_element.screenshot('capcha_temp/capcha.png')
    print("Скриншот капчи есть")
    
    #Вызываем функцию распознавания текста
    print("Отправка изображения на распознавание")
    text_result = recognize_captcha_via_prepostseo(driver, wait, 'capcha_temp/capcha.png')
    print(f"Текст получен: '{text_result}'")
    
    #возвращаемся на первую страницу
    driver.switch_to.window(demo_window)
    
    # 4. Поиск инпута и заполнение формы
    input_form = wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@id='imple-captcha-field' or contains(@class, 'input') or @type='text']")))
    input_form.clear()
    input_form.send_keys(text_result)
    
    #Нажимаем кнопку
    submit_button = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[@data-action='demo_action' or text()='Проверить']")))
    submit_button.click()
    print("Капча пройдена")



SyntaxError: incomplete input (1012750536.py, line 37)